# 01 - Preprocessing Pipeline

This notebook prepares the Restaurant ABSA dataset with the following pipeline:

1. Translate
2. Clean review text
3. Remove sentiment label
4. Standardize label values
5. Standardize label structure as multi-hot vectors
6. Clean data
7. Preprocess text for ML methods
8. Summarize and save data

Output:

- `data/Restaurant_ABSA_processed.csv`
- `Data/Restaurant_ABSA_processed.csv`
- `outputs/preprocessing/preprocessing_summary.csv`


## 0. Setup


In [ ]:
from pathlib import Path
from collections import Counter
import json
import re

import numpy as np
import pandas as pd

try:
    import nltk
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer
except ImportError as error:
    raise ImportError("Install nltk before running this notebook.") from error

try:
    import contractions
except ImportError:
    contractions = None

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "data").exists() or (candidate / "Data").exists():
        PROJECT_ROOT = candidate
        break

DATA_DIR = PROJECT_ROOT / "data"
LEGACY_DATA_DIR = PROJECT_ROOT / "Data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "preprocessing"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_XLSX_PATH = DATA_DIR / "Restaurant_ABSA.xlsx"
RAW_CSV_PATH = DATA_DIR / "Restaurant_ABSA.csv"
PROCESSED_PATH = DATA_DIR / "Restaurant_ABSA_processed.csv"
LEGACY_PROCESSED_PATH = LEGACY_DATA_DIR / "Restaurant_ABSA_processed.csv"

ASPECT_COLUMNS = ["food", "price", "service", "ambiance", "miscellaneous"]
SENTIMENT_COLUMN = "{Aspect category,   Sentiment Polarity}"

print("Project root:", PROJECT_ROOT)


In [ ]:
# NLTK data is required only for the ML-specific text field.
# If this fails in an offline environment, run once with internet:
# python -m nltk.downloader stopwords wordnet omw-1.4
for package in ["stopwords", "wordnet", "omw-1.4"]:
    try:
        nltk.data.find(f"corpora/{package}")
    except LookupError:
        nltk.download(package, quiet=True)

STOP_WORDS = set(stopwords.words("english"))
NEGATION_WORDS = {"no", "nor", "not", "never", "without", "cannot"}
STOP_WORDS_FOR_ML = STOP_WORDS - NEGATION_WORDS
LEMMATIZER = WordNetLemmatizer()


## 1. Load Raw Data


In [ ]:
if RAW_CSV_PATH.exists():
    raw_df = pd.read_csv(RAW_CSV_PATH)
    source_path = RAW_CSV_PATH
elif RAW_XLSX_PATH.exists():
    raw_df = pd.read_excel(RAW_XLSX_PATH)
    source_path = RAW_XLSX_PATH
else:
    raise FileNotFoundError("Missing Restaurant_ABSA.csv/xlsx in data/.")

print("Loaded:", source_path)
print("Shape:", raw_df.shape)
display(raw_df.head())


## 2. Step 1 — Translate


In [ ]:
def translate_reviews_if_needed(df: pd.DataFrame) -> pd.DataFrame:
    output = df.copy()

    if "review_en" in output.columns and output["review_en"].notna().any():
        output["review_en"] = output["review_en"].astype(str)
        print("Using existing review_en column; translation skipped.")
        return output

    if "Comment" not in output.columns:
        raise KeyError("Cannot translate because Comment column is missing.")

    try:
        from deep_translator import GoogleTranslator
    except ImportError as error:
        raise ImportError(
            "review_en is missing. Install deep-translator to translate Comment."
        ) from error

    translator = GoogleTranslator(source="auto", target="en")

    def translate_text(text):
        if pd.isna(text):
            return np.nan
        try:
            return translator.translate(str(text))
        except Exception:
            return str(text)

    output["review_en"] = output["Comment"].apply(translate_text)
    return output


translated_df = translate_reviews_if_needed(raw_df)
translated_df[["Comment", "review_en"]].head() if "Comment" in translated_df.columns else translated_df[["review_en"]].head()


## 3. Step 2 — Clean Review


In [ ]:
def expand_contractions(text: str) -> str:
    if contractions is not None:
        return contractions.fix(text)

    replacements = {
        "can't": "cannot",
        "won't": "will not",
        "n't": " not",
        "'re": " are",
        "'ve": " have",
        "'ll": " will",
        "'d": " would",
        "'m": " am",
    }
    for contraction, expansion in replacements.items():
        text = re.sub(re.escape(contraction), expansion, text, flags=re.IGNORECASE)
    return text


def clean_review_text(text) -> str:
    if pd.isna(text):
        return np.nan

    text = str(text)
    text = text.replace("\u200b", " ")
    text = expand_contractions(text)
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


cleaned_df = translated_df.copy()
cleaned_df["review_en"] = cleaned_df["review_en"].apply(clean_review_text)
cleaned_df[["review_en"]].head()


## 4. Step 3 — Remove Sentiment Label


In [ ]:
def parse_aspect_sentiment_pairs(value) -> list[tuple[str, str]]:
    if pd.isna(value):
        return []

    text = str(value)
    pairs = re.findall(
        r"\{\s*([^,{}]+?)\s*,\s*([^,{}]+?)\s*\}",
        text,
        flags=re.IGNORECASE,
    )
    return [(aspect.strip(), sentiment.strip()) for aspect, sentiment in pairs]


label_df = cleaned_df.copy()
if SENTIMENT_COLUMN not in label_df.columns:
    raise KeyError(f"Missing label column: {SENTIMENT_COLUMN}")

label_df["aspect_sentiment_pairs"] = label_df[SENTIMENT_COLUMN].apply(parse_aspect_sentiment_pairs)
label_df["aspect_raw"] = label_df["aspect_sentiment_pairs"].apply(
    lambda pairs: [aspect for aspect, _ in pairs]
)

# Sentiment polarity is intentionally removed because this task predicts aspect presence only.
label_df = label_df.drop(columns=[SENTIMENT_COLUMN, "aspect_sentiment_pairs"])

label_df[["review_en", "aspect_raw"]].head()


## 5. Step 4 — Standard Label Value


In [ ]:
ASPECT_NORMALIZATION = {
    "food": "food",
    "foods": "food",
    "price": "price",
    "prices": "price",
    "cost": "price",
    "service": "service",
    "serice": "service",
    "services": "service",
    "ambiance": "ambiance",
    "ambience": "ambiance",
    "environment": "ambiance",
    "misc": "miscellaneous",
    "miscellaneous": "miscellaneous",
    "miscelleneous": "miscellaneous",
    "mischellaneous": "miscellaneous",
}


def standardize_aspect_value(aspect: str) -> str | None:
    aspect = str(aspect).strip().lower()
    aspect = re.sub(r"[^a-z0-9\s]", " ", aspect)
    aspect = re.sub(r"\s+", " ", aspect).strip()
    return ASPECT_NORMALIZATION.get(aspect)


def standardize_aspect_list(aspects: list[str]) -> list[str]:
    standardized = []
    for aspect in aspects:
        normalized = standardize_aspect_value(aspect)
        if normalized is not None and normalized not in standardized:
            standardized.append(normalized)
    return standardized


standardized_df = label_df.copy()
standardized_df["aspect_list"] = standardized_df["aspect_raw"].apply(standardize_aspect_list)

raw_aspects = Counter(
    aspect
    for aspects in standardized_df["aspect_raw"]
    for aspect in aspects
)
standardized_aspects = Counter(
    aspect
    for aspects in standardized_df["aspect_list"]
    for aspect in aspects
)

print("Raw aspect values:", sorted(raw_aspects))
print("Standardized aspect values:", sorted(standardized_aspects))
standardized_df[["aspect_raw", "aspect_list"]].head()


## 6. Step 5 — Standard Label Structure


In [ ]:
multi_hot_df = standardized_df.copy()

for aspect in ASPECT_COLUMNS:
    multi_hot_df[aspect] = multi_hot_df["aspect_list"].apply(
        lambda aspects: int(aspect in aspects)
    )

multi_hot_df["num_aspects"] = multi_hot_df[ASPECT_COLUMNS].sum(axis=1)
multi_hot_df[["review_en", "aspect_list", "num_aspects", *ASPECT_COLUMNS]].head()


## 7. Step 6 — Clean Data


In [ ]:
before_cleaning = len(multi_hot_df)

clean_df = multi_hot_df.copy()
clean_df = clean_df.dropna(subset=["review_en"])
clean_df = clean_df[clean_df["review_en"].astype(str).str.strip().ne("")]
clean_df = clean_df[clean_df["num_aspects"] > 0]
clean_df = clean_df.drop_duplicates(subset=["review_en", *ASPECT_COLUMNS]).reset_index(drop=True)

after_cleaning = len(clean_df)
print("Rows before cleaning:", before_cleaning)
print("Rows after cleaning:", after_cleaning)
print("Removed rows:", before_cleaning - after_cleaning)

clean_df[["review_en", "num_aspects", *ASPECT_COLUMNS]].head()


## 8. Step 7 — Preprocessing For ML Method


In [ ]:
def preprocess_for_ml(text) -> str:
    if pd.isna(text):
        return np.nan

    tokens = str(text).split()
    tokens = [
        token
        for token in tokens
        if token not in STOP_WORDS_FOR_ML
    ]
    tokens = [
        LEMMATIZER.lemmatize(token)
        for token in tokens
    ]
    return " ".join(tokens)


processed_df = clean_df.copy()
processed_df["review_cleaned"] = processed_df["review_en"].apply(preprocess_for_ml)

final_columns = ["review_en", "review_cleaned", *ASPECT_COLUMNS]
processed_df = processed_df[final_columns].reset_index(drop=True)
processed_df.head()


## 9. Step 8 — Summarize And Save Data


In [ ]:
summary = {
    "source_path": str(source_path),
    "rows_raw": int(len(raw_df)),
    "rows_after_cleaning": int(len(processed_df)),
    "removed_rows": int(len(raw_df) - len(processed_df)),
    "duplicate_final_rows": int(processed_df.duplicated().sum()),
    "null_review_en": int(processed_df["review_en"].isna().sum()),
    "null_review_cleaned": int(processed_df["review_cleaned"].isna().sum()),
}

for aspect in ASPECT_COLUMNS:
    summary[f"{aspect}_positive"] = int(processed_df[aspect].sum())
    summary[f"{aspect}_positive_pct"] = float(processed_df[aspect].mean() * 100)

summary_df = pd.DataFrame([summary]).T.reset_index()
summary_df.columns = ["metric", "value"]
display(summary_df)


In [ ]:
label_distribution = (
    processed_df[ASPECT_COLUMNS]
    .sum()
    .rename_axis("aspect")
    .reset_index(name="positive_count")
)
label_distribution["positive_pct"] = (
    label_distribution["positive_count"] / len(processed_df) * 100
).round(2)

label_count_distribution = (
    processed_df[ASPECT_COLUMNS]
    .sum(axis=1)
    .value_counts()
    .sort_index()
    .rename_axis("label_count")
    .reset_index(name="review_count")
)

display(label_distribution)
display(label_count_distribution)


In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
LEGACY_DATA_DIR.mkdir(parents=True, exist_ok=True)

processed_df.to_csv(PROCESSED_PATH, index=False, encoding="utf-8")
processed_df.to_csv(LEGACY_PROCESSED_PATH, index=False, encoding="utf-8")
summary_df.to_csv(OUTPUT_DIR / "preprocessing_summary.csv", index=False, encoding="utf-8")
label_distribution.to_csv(OUTPUT_DIR / "label_distribution.csv", index=False, encoding="utf-8")
label_count_distribution.to_csv(OUTPUT_DIR / "label_count_distribution.csv", index=False, encoding="utf-8")

print("Saved processed data:", PROCESSED_PATH)
print("Saved legacy processed data:", LEGACY_PROCESSED_PATH)
print("Saved summary outputs:", OUTPUT_DIR)
